# AttritionGuard — Model Monitoring with Evidently AI

**Flow:** Load data → Data Drift Report → Column-level inspection → Test Suite with thresholds → Programmatic alert

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os

from evidently import Report, Dataset
from evidently.presets import DataDriftPreset, DataSummaryPreset

os.makedirs('reports', exist_ok=True)
print('Imports successful.')

Imports successful.


## 2. Load Reference and Current Data

- **Reference** — what the model was trained on (original distribution)
- **Current** — what the model is seeing in production 6 months later (drifted)

In [2]:
# Load datasets
reference = pd.read_csv('../data/processed/X_train.csv')
current   = pd.read_csv('../data/processed/X_drifted.csv')

print(f'Reference shape : {reference.shape}')
print(f'Current shape   : {current.shape}')

# Show the drift we introduced
features_to_check = ['OverTime', 'JobSatisfaction', 'MonthlyIncome', 'WorkLifeBalance']

print('\nFeature distribution comparison:')
print(f'{"Feature":<25} {"Reference Mean":>16} {"Current Mean":>14} {"Change":>10}')
print('-' * 68)
for f in features_to_check:
    ref_mean = reference[f].mean()
    cur_mean = current[f].mean()
    change   = ((cur_mean - ref_mean) / ref_mean) * 100
    print(f'{f:<25} {ref_mean:>16.2f} {cur_mean:>14.2f} {change:>+9.1f}%')

Reference shape : (1176, 30)
Current shape   : (1176, 30)

Feature distribution comparison:
Feature                     Reference Mean   Current Mean     Change
--------------------------------------------------------------------
OverTime                              0.29           0.59    +103.5%
JobSatisfaction                       2.72           1.95     -28.3%
MonthlyIncome                      6544.02        4907.65     -25.0%
WorkLifeBalance                       2.76           2.05     -25.6%


## 3. Wrap Data in Evidently Dataset

In [3]:
# Wrap DataFrames in Evidently Dataset objects
reference_ds = Dataset.from_pandas(reference)
current_ds   = Dataset.from_pandas(current)

print('Reference dataset:', reference_ds)
print('Current dataset  :', current_ds)

Reference dataset: <evidently.core.datasets.PandasDataset object at 0x12036ccd0>
Current dataset  : <evidently.core.datasets.PandasDataset object at 0x120d5ead0>


## 4. Run Data Drift Report

Evidently compares every feature's distribution between reference and current.
It automatically selects the right statistical test for each feature type:
- Numerical features → Kolmogorov-Smirnov test
- Categorical features → Chi-square test or Z-test

In [4]:
# Build and run the drift report
drift_report = Report([DataDriftPreset()])
data_drift_snapshot = drift_report.run(reference_ds, current_ds)

# Save as HTML — open in browser for full visual report
data_drift_snapshot.save_html('reports/data_drift_report.html')
print('HTML report saved: reports/data_drift_report.html')

HTML report saved: reports/data_drift_report.html


## 5. Inspect Results Programmatically

Extract the data drift analysis as a dict

In [5]:
results = data_drift_snapshot.dict()
metrics = results['metrics']

In [7]:
metrics[0]

{'id': '15e89f895b482f9b84ba7274ed18a106',
 'metric_name': 'DriftedColumnsCount(drift_share=0.5)',
 'config': {'type': 'evidently:metric_v2:DriftedColumnsCount',
  'drift_share': 0.5},
 'value': {'count': 4.0, 'share': 0.13333333333333333}}

In [12]:
# Get results as dictionary
results = data_drift_snapshot.dict()
metrics = results['metrics']

# Overall drift summary
overall     = metrics[0]
drift_count = overall['value']['count']
drift_share = overall['value']['share']

print('=' * 50)
print('OVERALL DRIFT SUMMARY')
print('=' * 50)
print(f'Drifted columns : {int(drift_count)} out of {len(reference.columns)}')
print(f'Share of drift  : {drift_share:.1%}')
print(f'Dataset drifted : {drift_share > 0.5}')

print()
print('=' * 50)
print('COLUMN-LEVEL DRIFT RESULTS')
print('=' * 50)
print(f'{"Feature":<30} {"Score":>12} {"Drifted":>10} {"Test Used"}')
print('-' * 80)

# 0 is for total count, so we start from 1 for column-level metrics
for m in metrics[1:]:
    name    = m['metric_name']
    score   = m['value']
    # Extract column name and test from metric name
    col     = name.split('column=')[1].split(',')[0] if 'column=' in name else name
    test    = name.split('method=')[1].split(',')[0] if 'method=' in name else 'N/A'
    # Evidently already handles drift detection internally
    # Trust the overall count from metrics[0] rather than re-implementing
    print(f'{col:<30} {score:>12.6f} {"See report":>10} {test}')

print()

OVERALL DRIFT SUMMARY
Drifted columns : 4 out of 30
Share of drift  : 13.3%
Dataset drifted : False

COLUMN-LEVEL DRIFT RESULTS
Feature                               Score    Drifted Test Used
--------------------------------------------------------------------------------
Age                                0.000000 See report Wasserstein distance (normed)
DailyRate                          0.000000 See report Wasserstein distance (normed)
DistanceFromHome                   0.000000 See report Wasserstein distance (normed)
HourlyRate                         0.000000 See report Wasserstein distance (normed)
MonthlyIncome                      0.469032 See report Wasserstein distance (normed)
MonthlyRate                        0.000000 See report Wasserstein distance (normed)
PercentSalaryHike                  0.000000 See report Wasserstein distance (normed)
TotalWorkingYears                  0.000000 See report Wasserstein distance (normed)
YearsAtCompany                     0.000000 Se

## 6. Data Summary Report

Shows descriptive statistics for both datasets side by side.

In [16]:
# Data summary — statistics for reference and current
summary_report = Report([DataSummaryPreset()])
summary_snap   = summary_report.run(reference_ds, current_ds)

summary_snap.save_html('reports/data_summary_report.html')
print('Summary report saved: reports/data_summary_report.html')

Summary report saved: reports/data_summary_report.html


## 7. Drift detection with thresholds

Thresholds are set for drift detection. If drift exceeds them, the test fails and triggers an alert.

In [17]:
4/30

0.13333333333333333

In [20]:
# Define thresholds
MAX_DRIFTED_COLUMNS = 3
MAX_DRIFT_SHARE = 0.10 # (0 to 1)

# Re-run drift report with explicit threshold tests
# DriftedColumnsCount — fail if more than 5 columns drift
# drift_share         — fail if more than 20% of columns drift

test_report = Report([
    DataDriftPreset(
        drift_share=0.5,   # column is drifted if p-value < 0.05
    )
])

test_snapshot = test_report.run(reference_ds, current_ds)

# Extract results
test_results  = test_snapshot.dict()['metrics']
overall       = test_results[0]
drifted_count = int(overall['value']['count'])
drifted_share = overall['value']['share']



# Evaluate
count_test_passed = drifted_count <= MAX_DRIFTED_COLUMNS
share_test_passed = drifted_share <= MAX_DRIFT_SHARE
all_passed        = count_test_passed and share_test_passed

print('=' * 50)
print('TEST SUITE RESULTS')
print('=' * 50)
print(f'Test 1 — Drifted columns <= {MAX_DRIFTED_COLUMNS}')
print(f'  Actual  : {drifted_count}')
print(f'  Result  : {"PASSED" if count_test_passed else "FAILED"}')
print()
print(f'Test 2 — Drift share <= {MAX_DRIFT_SHARE:.0%}')
print(f'  Actual  : {drifted_share:.1%}')
print(f'  Result  : {"PASSED" if share_test_passed else "FAILED"}')
print()
print(f'Overall : {"ALL PASSED" if all_passed else "TESTS FAILED"}')

TEST SUITE RESULTS
Test 1 — Drifted columns <= 3
  Actual  : 4
  Result  : FAILED

Test 2 — Drift share <= 10%
  Actual  : 13.3%
  Result  : FAILED

Overall : TESTS FAILED
